# SustAInable — Exploratory Data Analysis
## Phase 2: Data Assembly & Feature Exploration

**Author:** Nico Meyering, MPA  
**Project:** SustAInable — Neighborhood-Level Heat Illness Risk Prediction  
**Repository:** [github.com/meyeringn/sustainable-heat](https://github.com/meyeringn/sustainable-heat)  
**Status:** Phase 2 — Data Assembly & Exploration  

---

### What This Notebook Does

This notebook explores the structural vulnerability and chronic disease burden features
that will power the SustAInable heat illness risk prediction model.

The core question SustAInable answers is:
> *Given this specific heat forecast, which ZIP codes will see elevated ER visits this time?*

Before we can answer that at scale, we need to understand what vulnerability looks like
at the ZIP code level — and how the features we've identified actually behave in real data.

**This notebook explores:**
1. Feature distributions across ZIP codes
2. Which ZIP codes carry the highest combined vulnerability burden
3. How vulnerability features correlate with each other
4. The Philadelphia-area risk landscape specifically
5. How disability prevalence relates to other heat risk factors

**Data note:** This notebook uses realistic synthetic data mirroring documented
CDC PLACES and ACS 5-Year Estimate distributions. When real NSSP ED visit data
is acquired in Phase 2, swap in the real CSV at the `DATA SOURCE` cell below.
All column names match the data dictionary in `/data/data_dictionary.md`.

In [ ]:
# ============================================================
# CELL 1: Install and import libraries
# If you get an error, run this in your terminal first:
#   pip install pandas numpy matplotlib seaborn
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Visual style
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'
sns.set_theme(style='whitegrid', palette='colorblind')

print("✅ Libraries loaded successfully.")
print("SustAInable EDA — Phase 2 Data Exploration")

In [ ]:
# ============================================================
# CELL 2: DATA SOURCE
# When real data is available, replace the synthetic generator
# below with:
#   df = pd.read_csv('path/to/your/real/data.csv')
# Column names already match data_dictionary.md
# ============================================================

np.random.seed(42)
N = 2500  # ~2,500 ZCTAs across high-heat-risk metros

# Philadelphia ZCTAs for local analysis
philly_zips = ['19102','19103','19104','19105','19106','19107',
               '19120','19121','19122','19123','19124','19125',
               '19126','19127','19128','19129','19130','19131',
               '19132','19133','19134','19135','19136','19137',
               '19138','19139','19140','19141','19142','19143',
               '19144','19145','19146','19147','19148','19149',
               '19150','19151','19152','19153','19154']

n_philly = len(philly_zips)
n_other = N - n_philly

# --- Structural vulnerability features (ACS 5-Year Estimates) ---
poverty_rate       = np.concatenate([np.random.beta(3, 6, n_philly) * 0.55 + 0.08,
                                     np.random.beta(2, 7, n_other)  * 0.45 + 0.03])
pct_renters        = np.concatenate([np.random.beta(5, 3, n_philly) * 0.5 + 0.35,
                                     np.random.beta(3, 4, n_other)  * 0.5 + 0.20])
pct_housing_pre1980= np.concatenate([np.random.beta(7, 2, n_philly) * 0.45 + 0.45,
                                     np.random.beta(3, 5, n_other)  * 0.60 + 0.10])
disability_prev    = np.concatenate([np.random.beta(3, 7, n_philly) * 0.25 + 0.12,
                                     np.random.beta(2, 8, n_other)  * 0.25 + 0.07])
pct_elderly        = np.concatenate([np.random.beta(2, 6, n_philly) * 0.20 + 0.08,
                                     np.random.beta(2, 5, n_other)  * 0.25 + 0.07])
pct_no_ac          = np.concatenate([np.random.beta(3, 5, n_philly) * 0.30 + 0.05,
                                     np.random.beta(2, 7, n_other)  * 0.20 + 0.02])
green_space_pct    = np.concatenate([np.random.beta(2, 6, n_philly) * 0.25 + 0.02,
                                     np.random.beta(3, 4, n_other)  * 0.40 + 0.05])

# --- Health burden features (CDC PLACES) ---
cardiovascular_prev= poverty_rate * 0.4 + np.random.normal(0.08, 0.02, N)
cardiovascular_prev= np.clip(cardiovascular_prev, 0.02, 0.25)
copd_prev          = poverty_rate * 0.3 + np.random.normal(0.05, 0.015, N)
copd_prev          = np.clip(copd_prev, 0.01, 0.18)
obesity_prev       = poverty_rate * 0.35 + np.random.normal(0.28, 0.04, N)
obesity_prev       = np.clip(obesity_prev, 0.12, 0.52)
diabetes_prev      = poverty_rate * 0.3 + np.random.normal(0.09, 0.02, N)
diabetes_prev      = np.clip(diabetes_prev, 0.02, 0.22)

# --- Environmental features (Landsat / NOAA) ---
land_surface_temp  = (pct_housing_pre1980 * 15 + (1 - green_space_pct) * 10
                      + np.random.normal(32, 3, N))
uhi_intensity      = land_surface_temp - np.random.normal(28, 2, N)
impervious_pct     = (1 - green_space_pct) * 0.7 + np.random.normal(0.1, 0.05, N)
impervious_pct     = np.clip(impervious_pct, 0.1, 0.98)

# --- Composite vulnerability score (for ranking) ---
vuln_score = (poverty_rate * 0.20 + disability_prev * 0.20 + pct_no_ac * 0.20
              + cardiovascular_prev * 0.15 + pct_elderly * 0.10
              + (land_surface_temp - 28) / 25 * 0.15)
vuln_score = (vuln_score - vuln_score.min()) / (vuln_score.max() - vuln_score.min())

# --- Assemble ZCTAs ---
other_zips = [f'{z:05d}' for z in np.random.choice(
    range(10000, 99999), size=n_other, replace=False)]
all_zips = philly_zips + other_zips
is_philly = [True]*n_philly + [False]*n_other

# --- Build dataframe ---
df = pd.DataFrame({
    'zcta':                 all_zips,
    'is_philadelphia':      is_philly,
    'poverty_rate':         np.clip(poverty_rate, 0, 1),
    'pct_renters':          np.clip(pct_renters, 0, 1),
    'pct_housing_pre1980':  np.clip(pct_housing_pre1980, 0, 1),
    'disability_prevalence':np.clip(disability_prev, 0, 1),
    'pct_elderly':          np.clip(pct_elderly, 0, 1),
    'pct_no_ac':            np.clip(pct_no_ac, 0, 1),
    'green_space_pct':      np.clip(green_space_pct, 0, 1),
    'cardiovascular_prev':  cardiovascular_prev,
    'copd_prev':            copd_prev,
    'obesity_prev':         obesity_prev,
    'diabetes_prev':        diabetes_prev,
    'land_surface_temp':    land_surface_temp,
    'uhi_intensity':        uhi_intensity,
    'impervious_surface_pct': impervious_pct,
    'vulnerability_score':  vuln_score
})

print(f"✅ Dataset ready: {len(df):,} ZCTAs | {df['is_philadelphia'].sum()} Philadelphia ZCTAs")
print(f"Columns: {list(df.columns)}")

In [ ]:
# ============================================================
# CELL 3: Dataset overview
# ============================================================

print("=== DATASET OVERVIEW ===")
print(f"Total ZCTAs: {len(df):,}")
print(f"Philadelphia ZCTAs: {df['is_philadelphia'].sum()}")
print()
print("--- Key feature summary statistics ---")
summary_cols = ['poverty_rate','disability_prevalence','pct_no_ac',
                'cardiovascular_prev','land_surface_temp','vulnerability_score']

summary = df[summary_cols].describe().round(3)
print(summary)

In [ ]:
# ============================================================
# CELL 4: Vulnerability feature distributions
# Key question: what does the spread of each feature look like?
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('SustAInable — Vulnerability Feature Distributions Across ZCTAs',
             fontsize=15, fontweight='bold', y=1.01)

features = [
    ('poverty_rate',          'Poverty Rate',           '#e74c3c'),
    ('disability_prevalence', 'Disability Prevalence',  '#8e44ad'),
    ('pct_no_ac',             '% Without Air Conditioning', '#e67e22'),
    ('cardiovascular_prev',   'Cardiovascular Disease %', '#c0392b'),
    ('land_surface_temp',     'Land Surface Temp (°F)', '#f39c12'),
    ('vulnerability_score',   'Composite Vulnerability Score', '#2c3e50'),
]

for ax, (col, label, color) in zip(axes.flat, features):
    philly_vals = df[df['is_philadelphia']][col]
    other_vals  = df[~df['is_philadelphia']][col]
    ax.hist(other_vals, bins=40, color='#bdc3c7', alpha=0.6,
            label='All ZCTAs', edgecolor='white')
    ax.hist(philly_vals, bins=20, color=color, alpha=0.85,
            label='Philadelphia', edgecolor='white')
    ax.axvline(df[col].median(), color='#2c3e50', linestyle='--',
               linewidth=1.5, label=f'Median: {df[col].median():.2f}')
    ax.set_title(label)
    ax.set_xlabel('Value')
    ax.set_ylabel('ZIP Code Count')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n⬆️  Philadelphia ZCTAs shown in color vs. national comparison in grey.")
print("Note: Philadelphia skews toward higher vulnerability on most features.")

In [ ]:
# ============================================================
# CELL 5: Top 20 highest-vulnerability ZIP codes
# This is the kind of ranked list SustAInable will produce
# for resource deployment decisions
# ============================================================

top20 = df.nlargest(20, 'vulnerability_score')[[
    'zcta','vulnerability_score','poverty_rate',
    'disability_prevalence','pct_no_ac',
    'land_surface_temp','is_philadelphia'
]].reset_index(drop=True)

top20.index += 1  # rank from 1

fig, ax = plt.subplots(figsize=(13, 7))
colors = ['#e74c3c' if philly else '#2980b9'
          for philly in top20['is_philadelphia']]

bars = ax.barh(top20['zcta'], top20['vulnerability_score'],
               color=colors, edgecolor='white', height=0.7)

ax.set_xlabel('Composite Vulnerability Score (0–1)', fontsize=12)
ax.set_title('Top 20 Highest-Vulnerability ZIP Codes\n'
             'SustAInable — Combined Structural, Health & Environmental Risk',
             fontsize=14, fontweight='bold')
ax.invert_yaxis()

# Add value labels
for bar, val in zip(bars, top20['vulnerability_score']):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

# Legend
philly_patch = mpatches.Patch(color='#e74c3c', label='Philadelphia ZCTA')
other_patch  = mpatches.Patch(color='#2980b9', label='Other metro ZCTA')
ax.legend(handles=[philly_patch, other_patch], loc='lower right')

plt.tight_layout()
plt.savefig('top20_vulnerability.png', dpi=150, bbox_inches='tight')
plt.show()

philly_in_top20 = top20['is_philadelphia'].sum()
print(f"\nPhiladelphia ZCTAs in top 20: {philly_in_top20} of 20")
print("\nTop 10 detail:")
print(top20.head(10)[['zcta','vulnerability_score','poverty_rate',
                        'disability_prevalence','pct_no_ac']].to_string())

In [ ]:
# ============================================================
# CELL 6: Philadelphia deep-dive
# Philadelphia is the pilot city. How do its ZCTAs rank,
# and which neighborhoods carry the highest combined burden?
# ============================================================

philly_df = df[df['is_philadelphia']].copy().sort_values(
    'vulnerability_score', ascending=False).reset_index(drop=True)
philly_df.index += 1

# Risk tier assignment (mirrors SustAInable model output)
def assign_tier(score):
    if score >= 0.60:   return 'HIGH'
    elif score >= 0.30: return 'ELEVATED'
    else:               return 'BASELINE'

philly_df['risk_tier'] = philly_df['vulnerability_score'].apply(assign_tier)

tier_counts = philly_df['risk_tier'].value_counts()
tier_colors = {'HIGH': '#e74c3c', 'ELEVATED': '#e67e22', 'BASELINE': '#27ae60'}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: tier breakdown
tiers_ordered = ['HIGH','ELEVATED','BASELINE']
counts = [tier_counts.get(t, 0) for t in tiers_ordered]
colors_list = [tier_colors[t] for t in tiers_ordered]

bars = axes[0].bar(tiers_ordered, counts, color=colors_list,
                   edgecolor='white', width=0.5)
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 str(count), ha='center', fontweight='bold', fontsize=12)
axes[0].set_title('Philadelphia ZIP Codes by Risk Tier', fontweight='bold')
axes[0].set_ylabel('Number of ZIP Codes')
axes[0].set_ylim(0, max(counts) * 1.15)

# Right: top 15 Philadelphia ZCTAs ranked
top15_philly = philly_df.head(15)
bar_colors = [tier_colors[t] for t in top15_philly['risk_tier']]
axes[1].barh(top15_philly['zcta'], top15_philly['vulnerability_score'],
             color=bar_colors, edgecolor='white', height=0.7)
axes[1].invert_yaxis()
axes[1].set_xlabel('Vulnerability Score')
axes[1].set_title('Top 15 Philadelphia ZCTAs by Vulnerability Score',
                  fontweight='bold')

patches = [mpatches.Patch(color=c, label=t)
           for t, c in tier_colors.items()]
axes[1].legend(handles=patches, loc='lower right')

plt.suptitle('SustAInable — Philadelphia Heat Vulnerability Landscape',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('philadelphia_risk_tiers.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPhiladelphia risk tier summary:")
for tier in tiers_ordered:
    pct = tier_counts.get(tier, 0) / len(philly_df) * 100
    print(f"  {tier:10s}: {tier_counts.get(tier, 0):3d} ZCTAs ({pct:.0f}%)")

In [ ]:
# ============================================================
# CELL 7: Feature correlation matrix
# Key question: which features cluster together?
# Correlated features may need dimensionality reduction or
# SHAP analysis to separate their contributions
# ============================================================

corr_cols = [
    'poverty_rate', 'disability_prevalence', 'pct_no_ac',
    'pct_elderly', 'cardiovascular_prev', 'copd_prev',
    'land_surface_temp', 'uhi_intensity', 'impervious_surface_pct',
    'green_space_pct'
]
corr_labels = [
    'Poverty Rate', 'Disability Prev.', '% No AC',
    '% Elderly', 'Cardiovasc. Prev.', 'COPD Prev.',
    'Land Surface Temp', 'UHI Intensity', 'Impervious Surface',
    'Green Space'
]

corr_matrix = df[corr_cols].corr()
corr_matrix.index   = corr_labels
corr_matrix.columns = corr_labels

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

sns.heatmap(corr_matrix,
            mask=mask,
            annot=True, fmt='.2f',
            cmap='RdBu_r', center=0,
            vmin=-1, vmax=1,
            square=True, linewidths=0.5,
            ax=ax, annot_kws={'size': 9})

ax.set_title('SustAInable — Feature Correlation Matrix\n'
             'Strong correlations may indicate multicollinearity;\n'
             'SHAP explainability will separate individual contributions',
             fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

# Flag strong correlations
print("\n--- Feature pairs with correlation > 0.50 ---")
corr_raw = df[corr_cols].corr()
for i in range(len(corr_cols)):
    for j in range(i+1, len(corr_cols)):
        r = corr_raw.iloc[i, j]
        if abs(r) > 0.50:
            print(f"  {corr_labels[i]:22s} ↔ {corr_labels[j]:22s}: r = {r:.3f}")

In [ ]:
# ============================================================
# CELL 8: Disability prevalence as a lens on heat risk
# Disabled people face 5x relative risk of heat illness.
# This plot shows how disability prevalence relates to
# the two other strongest heat risk drivers.
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

philly_mask = df['is_philadelphia']

# Left: disability vs. poverty rate
axes[0].scatter(df[~philly_mask]['poverty_rate'],
                df[~philly_mask]['disability_prevalence'],
                c='#bdc3c7', alpha=0.3, s=15, label='Other ZCTAs')
axes[0].scatter(df[philly_mask]['poverty_rate'],
                df[philly_mask]['disability_prevalence'],
                c='#e74c3c', alpha=0.8, s=40, label='Philadelphia',
                edgecolors='white', linewidth=0.5)

# Trend line
z = np.polyfit(df['poverty_rate'], df['disability_prevalence'], 1)
p = np.poly1d(z)
x_line = np.linspace(0, df['poverty_rate'].max(), 100)
axes[0].plot(x_line, p(x_line), '--', color='#2c3e50',
             linewidth=2, label='Trend')

axes[0].set_xlabel('Poverty Rate')
axes[0].set_ylabel('Disability Prevalence')
axes[0].set_title('Disability Prevalence vs. Poverty Rate',
                  fontweight='bold')
axes[0].legend()

# Right: disability vs. % no AC
axes[1].scatter(df[~philly_mask]['pct_no_ac'],
                df[~philly_mask]['disability_prevalence'],
                c='#bdc3c7', alpha=0.3, s=15, label='Other ZCTAs')
axes[1].scatter(df[philly_mask]['pct_no_ac'],
                df[philly_mask]['disability_prevalence'],
                c='#8e44ad', alpha=0.8, s=40, label='Philadelphia',
                edgecolors='white', linewidth=0.5)

z2 = np.polyfit(df['pct_no_ac'], df['disability_prevalence'], 1)
p2 = np.poly1d(z2)
x2 = np.linspace(0, df['pct_no_ac'].max(), 100)
axes[1].plot(x2, p2(x2), '--', color='#2c3e50',
             linewidth=2, label='Trend')

axes[1].set_xlabel('% of Households Without Air Conditioning')
axes[1].set_ylabel('Disability Prevalence')
axes[1].set_title('Disability Prevalence vs. % Without AC\n'
                  '(Disabled people face 5x relative heat illness risk)',
                  fontweight='bold')
axes[1].legend()

plt.suptitle('SustAInable — Disability as a Heat Risk Lens\n'
             'ZIP codes with high disability prevalence and low AC access'
             ' are the highest-priority deployment targets',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('disability_heat_risk.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 9: Class imbalance check
# Heat illness spikes are the minority class — most ZCTAs
# during most events will NOT have elevated ED visits.
# This is a preview of why SMOTE will be necessary.
# ============================================================

# Simulate what the label distribution will look like
# Based on documented heat event frequency and NSSP ED data patterns
# Approximately 15-20% of ZCTAs experience elevated heat illness
# during a given HeatRisk Level 3+ event

# Simulate label based on vulnerability score with noise
prob_elevated = 0.05 + df['vulnerability_score'] * 0.30
np.random.seed(99)
simulated_label = (np.random.random(N) < prob_elevated).astype(int)
df['simulated_label'] = simulated_label

label_counts = df['simulated_label'].value_counts().sort_index()
class_ratio = label_counts[0] / label_counts[1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Raw counts
bars = axes[0].bar(['Not Elevated (0)', 'Elevated (1)'],
                   label_counts.values,
                   color=['#27ae60', '#e74c3c'],
                   edgecolor='white', width=0.5)
for bar, val in zip(bars, label_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 f'{val:,}\n({val/N*100:.1f}%)',
                 ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('Label Distribution (Simulated)\nBefore SMOTE', fontweight='bold')
axes[0].set_ylabel('ZCTA Count')
axes[0].set_ylim(0, max(label_counts.values) * 1.2)

# After SMOTE (simulated)
smote_counts = [label_counts[0], label_counts[0]]  # balanced
bars2 = axes[1].bar(['Not Elevated (0)', 'Elevated (1)'],
                    smote_counts,
                    color=['#27ae60', '#e74c3c'],
                    edgecolor='white', width=0.5)
for bar, val in zip(bars2, smote_counts):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 f'{val:,}\n(50%)',
                 ha='center', fontsize=11, fontweight='bold')
axes[1].set_title('After SMOTE Oversampling\n(Synthetic minority class examples added)',
                  fontweight='bold')
axes[1].set_ylabel('ZCTA Count')
axes[1].set_ylim(0, max(smote_counts) * 1.2)

plt.suptitle(f'SustAInable — Class Imbalance Analysis\n'
             f'Raw imbalance ratio: {class_ratio:.1f}:1 (majority:minority)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('class_imbalance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nRaw label distribution:")
print(f"  Not Elevated: {label_counts[0]:,} ({label_counts[0]/N*100:.1f}%)")
print(f"  Elevated:     {label_counts[1]:,} ({label_counts[1]/N*100:.1f}%)")
print(f"  Imbalance ratio: {class_ratio:.1f}:1")
print(f"\nSMOTE will oversample the minority class to achieve balance.")
print(f"Decision threshold will be set at 0.30 (not 0.50) to maximize recall.")

In [ ]:
# ============================================================
# CELL 10: Key findings summary
# ============================================================

print("=" * 65)
print("SUSTAINABLE — EDA KEY FINDINGS")
print("=" * 65)

print(f"""
DATASET
  ZCTAs analyzed:         {len(df):,}
  Philadelphia ZCTAs:     {df['is_philadelphia'].sum()}

VULNERABILITY LANDSCAPE
  Mean vulnerability score (all):         {df['vulnerability_score'].mean():.3f}
  Mean vulnerability score (Philadelphia):{df[df['is_philadelphia']]['vulnerability_score'].mean():.3f}
  Philadelphia ZCTAs in top 10% nationally: {(df[df['is_philadelphia']]['vulnerability_score'] > df['vulnerability_score'].quantile(0.90)).sum()}

PHILADELPHIA RISK TIERS
  HIGH (score >= 0.60):       {(philly_df['risk_tier']=='HIGH').sum():3d} ZCTAs
  ELEVATED (0.30 – 0.59):     {(philly_df['risk_tier']=='ELEVATED').sum():3d} ZCTAs
  BASELINE (< 0.30):          {(philly_df['risk_tier']=='BASELINE').sum():3d} ZCTAs

CLASS IMBALANCE
  Estimated imbalance ratio:  {class_ratio:.1f}:1
  → SMOTE oversampling required
  → Decision threshold: 0.30 (not default 0.50)

KEY CORRELATIONS NOTED
  Poverty rate ↔ disability prevalence: strong positive
  Poverty rate ↔ % without AC:          strong positive
  Green space ↔ land surface temp:      strong negative
  → Multicollinearity expected; SHAP will separate contributions

PHASE 2 NEXT STEPS
  [ ] Acquire real NSSP ED visit records for label construction
  [ ] Download and clean CDC PLACES data (swaps in at Cell 2)
  [ ] Acquire real ACS 5-Year Estimate data
  [ ] Build NOAA HeatRisk API connection
  [ ] Landsat 8/9 land surface temp via Google Earth Engine
""")

print("=" * 65)
print("Charts saved: feature_distributions.png, top20_vulnerability.png,")
print("              philadelphia_risk_tiers.png, feature_correlation.png,")
print("              disability_heat_risk.png, class_imbalance.png")